In [1]:
import os
import cv2
import numpy as np
import torch
import torch.nn as nn
from transformers import AutoModel
from torchvision import transforms
from ultralytics import YOLO
import supervision as sv
from sahi import AutoDetectionModel
from sahi.predict import get_sliced_prediction
from collections import defaultdict
from datetime import datetime
import csv
import pandas as pd

In [2]:
# ──── Paths ────
YOLO_MODEL_PATH   = "Trained_Models/detection_mid.pt"
DINOV2_MODEL_PATH = "Trained_Models/vit_small.pt"
IMAGE_DIR         = "data/images"
OUTPUT_CSV        = f"results_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv"

# ──── Classes ────
ACTION_CLASSES = ["background", "fanning", "trophallaxis"]  # must match training order
CONFIDENCE_THRESHOLD = 0.6  # ViT predictions below this will show as "uncertain"

# ──── Inference settings ────
FPS        = 10
CROP_PAD   = 0.25
BATCH_SIZE = 1  # single image at a time for real-time display

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
print(f"Action classes: {ACTION_CLASSES}")
print(f"Saving results to: {OUTPUT_CSV}")

Using device: cuda
Action classes: ['background', 'fanning', 'trophallaxis']
Saving results to: results_20260320_142252.csv


In [3]:
# Load YOLO detection model
yolo_model = YOLO(YOLO_MODEL_PATH)

# Wrap with SAHI for sliced inference
sahi_model = AutoDetectionModel.from_pretrained(
    model_type='ultralytics',
    model=yolo_model,
    confidence_threshold=0.45,
    device='cuda:0'
)

print("YOLO + SAHI loaded successfully")

YOLO + SAHI loaded successfully


In [4]:
class DINOv2Classifier(nn.Module):
    def __init__(self, backbone, num_classes, hidden_size=384):
        super().__init__()
        self.backbone = backbone
        self.classifier = nn.Sequential(
            nn.Linear(hidden_size, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes)
        )
    
    def forward(self, x):
        outputs = self.backbone(x)
        cls_token = outputs.last_hidden_state[:, 0]
        return self.classifier(cls_token)
# Load backbone and restore trained weights
dinov2_backbone = AutoModel.from_pretrained("facebook/dinov2-small")
action_model = DINOv2Classifier(dinov2_backbone, num_classes=len(ACTION_CLASSES)).to(device)
action_model.load_state_dict(torch.load(DINOV2_MODEL_PATH, map_location=device))
action_model.eval()

print(f"DINOv2 classifier loaded from {DINOV2_MODEL_PATH}")
print(f"Classifying into: {ACTION_CLASSES}")

Loading weights:   0%|          | 0/223 [00:00<?, ?it/s]

DINOv2 classifier loaded from Trained_Models/vit_small.pt
Classifying into: ['background', 'fanning', 'trophallaxis']


In [5]:
# Preprocessing transform — must match training
preprocess = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

In [6]:
def classify_crop(crop):
    """
    Takes a bee crop and returns predicted action label and confidence.
    
    Args:
        crop: numpy array in BGR format (from OpenCV)
    
    Returns:
        label:      predicted action string or "uncertain"
        confidence: float between 0 and 1
    """
    # Convert BGR to RGB
    crop_rgb = cv2.cvtColor(crop, cv2.COLOR_BGR2RGB)
    
    # Preprocess and add batch dimension
    tensor = preprocess(crop_rgb).unsqueeze(0).to(device)
    
    with torch.no_grad():
        outputs = action_model(tensor)
        probs = torch.softmax(outputs, dim=1)
        confidence, pred_idx = probs.max(dim=1)
    
    confidence = confidence.item()
    pred_idx   = pred_idx.item()
    
    if confidence < CONFIDENCE_THRESHOLD:
        return "uncertain", confidence
    
    return ACTION_CLASSES[pred_idx], confidence

In [7]:
# Quick test with a dummy black image
dummy_crop = np.zeros((100, 100, 3), dtype=np.uint8)
label, conf = classify_crop(dummy_crop)
print(f"Test prediction: {label} ({conf:.2f})")
print("classify_crop ready!")

Test prediction: uncertain (0.56)
classify_crop ready!


In [8]:
def extract_crop(frame, xyxy, padding=0.25):
    """Extract padded crop from frame using bounding box."""
    h, w = frame.shape[:2]
    x1, y1, x2, y2 = map(int, xyxy)
    
    pad_x = int((x2 - x1) * padding)
    pad_y = int((y2 - y1) * padding)
    x1 = max(0, x1 - pad_x)
    y1 = max(0, y1 - pad_y)
    x2 = min(w, x2 + pad_x)
    y2 = min(h, y2 + pad_y)
    
    crop = frame[y1:y2, x1:x2]
    return crop if crop.size > 0 else None
def log_result(frame_idx, img_name, action, confidence, xyxy):
    """Append a single detection result to the CSV."""
    x1, y1, x2, y2 = map(int, xyxy)
    with open(OUTPUT_CSV, "a", newline="") as f:
        writer = csv.writer(f)
        writer.writerow([frame_idx, img_name, action, f"{confidence:.4f}", x1, y1, x2, y2])
# Initialize CSV file
with open(OUTPUT_CSV, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["frame", "image_name", "action", "confidence", "x1", "y1", "x2", "y2"])


In [9]:
# Colour map per action for bounding boxes
ACTION_COLORS = {
    "fanning":      sv.Color(r=0,   g=255, b=100),  # green
    "trophallaxis": sv.Color(r=255, g=100, b=0),    # orange
    "background":   sv.Color(r=150, g=150, b=150),  # grey
    "uncertain":    sv.Color(r=255, g=0,   b=0),    # red
}

box_annotator   = sv.BoxAnnotator(thickness=2)
label_annotator = sv.LabelAnnotator(text_thickness=1, text_scale=0.5)

print("Annotators ready")

Annotators ready


In [10]:
images = sorted([f for f in os.listdir(IMAGE_DIR) if f.endswith((".jpg"))])
delay = int(1000 / FPS)

print(f"Processing {len(images)} frames...")

Processing 6200 frames...


In [11]:
for frame_idx, img_name in enumerate(images[:]):
    img_path = os.path.join(IMAGE_DIR, img_name)
    frame = cv2.imread(img_path)
    
    if frame is None:
        continue
    
    # ── Detection with SAHI ────────────────────────
    result = get_sliced_prediction(
        img_path, sahi_model,
        slice_height=640, slice_width=640,
        overlap_height_ratio=0.25, overlap_width_ratio=0.25,
        verbose=0
    )
    
    boxes, class_ids, confidences = [], [], []
    for pred in result.object_prediction_list:
        boxes.append(pred.bbox.to_xyxy())
        class_ids.append(pred.category.id)
        confidences.append(pred.score.value)
    
    if len(boxes) > 0:
        detections = sv.Detections(
            xyxy=np.array(boxes),
            class_id=np.array(class_ids),
            confidence=np.array(confidences)
        )
    else:
        detections = sv.Detections.empty()
    
    # ── Classification + Logging ───────────────────
    labels = []
    colors = []
    
    for i, class_id in enumerate(detections.class_id):
        crop = extract_crop(frame, detections.xyxy[i], padding=CROP_PAD)
        
        if crop is not None:
            action, confidence = classify_crop(crop)
        else:
            action, confidence = "uncertain", 0.0
        
        # Log to CSV
        log_result(frame_idx, img_name, action, confidence, detections.xyxy[i])
        
        # Build label and color per detection
        det_conf = detections.confidence[i]
        labels.append(f"#bee({det_conf:.2f}) - {action}({confidence:.2f})")
        colors.append(ACTION_COLORS.get(action, ACTION_COLORS["uncertain"]))
    
    # ── Annotation ─────────────────────────────────
    annotated = box_annotator.annotate(scene=frame.copy(), detections=detections)
    annotated = label_annotator.annotate(scene=annotated, detections=detections, labels=labels)
    
    cv2.putText(annotated, f"Frame: {frame_idx} | {img_name}",
                (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
    
    cv2.imshow("Bee Detection + Action Recognition", annotated)
    
    key = cv2.waitKey(delay)
    if key == 27:    # ESC to quit
        break
    elif key == 32:  # SPACE to pause
        cv2.waitKey(0)

cv2.destroyAllWindows()
print(f"Done! Results saved to {OUTPUT_CSV}")

Done! Results saved to results_20260320_142252.csv


In [12]:
df = pd.read_csv(OUTPUT_CSV)
print(f"Total detections logged: {len(df)}")
print(f"\nAction distribution:")
print(df["action"].value_counts())
print(f"\nSample rows:")
print(df.head(10))

Total detections logged: 257

Action distribution:
action
fanning         221
trophallaxis     25
uncertain        11
Name: count, dtype: int64

Sample rows:
   frame         image_name   action  confidence    x1    y1    x2    y2
0      0  20230609a1002.jpg  fanning      0.9997    65   702   129   786
1      0  20230609a1002.jpg  fanning      1.0000   629   861   694   942
2      0  20230609a1002.jpg  fanning      0.9997  1258   515  1317   592
3      0  20230609a1002.jpg  fanning      1.0000   726   429   798   498
4      0  20230609a1002.jpg  fanning      1.0000   975   534  1037   584
5      0  20230609a1002.jpg  fanning      0.9749   766   969   822  1056
6      0  20230609a1002.jpg  fanning      0.9928  1082  1024  1122  1079
7      1  20230609a1008.jpg  fanning      1.0000   328   561   383   623
8      1  20230609a1008.jpg  fanning      1.0000  1079   636  1136   726
9      1  20230609a1008.jpg  fanning      0.9987  1305   792  1346   863
